In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from pathlib import Path

# Prefer an explicit env var; otherwise find the repo root from the notebook cwd.
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "track2").exists() and (candidate / "outputs").exists():
            return candidate
    fallback = Path("/root/autodl-tmp/nlpcc_task2")
    return fallback if fallback.exists() else Path.cwd()

ROOT_DIR = Path(os.environ.get("NLPCC_TASK2_ROOT", find_project_root())).resolve()

TRAIN_DATA_FILE = ROOT_DIR / "data" / "track2" / "sft_train.jsonl"
DEV_DATA_FILE = ROOT_DIR / "data" / "track2" / "sft_dev.jsonl"

SFT_MODEL_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "sft"
TRACK1_MODEL_DIR = ROOT_DIR / "outputs" / "track1" / "encoder" / "bert_scl_optimized"

OUTPUTS_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "ppo"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT_DIR:", ROOT_DIR)
print("SFT_MODEL_DIR:", SFT_MODEL_DIR)


ROOT_DIR: /root/autodl-tmp/nlpcc_task2
SFT_MODEL_DIR: /root/autodl-tmp/nlpcc_task2/outputs/track2/qwen35_9b/sft


In [ ]:
TRACK1_BASE_MODEL = "bert-base-uncased"
TRACK2_BASE_MODEL = os.environ.get("TRACK2_BASE_MODEL", "Qwen/Qwen3.5-9B")
LOCAL_TRACK2_BASE_MODEL = Path(os.environ.get("LOCAL_TRACK2_BASE_MODEL", "/root/autodl-fs/nlpcc_models/Qwen3.5-9B"))

# Keep this off by default so this test notebook always uses the server-local base model.
ALLOW_REMOTE_MODEL_DOWNLOAD = os.environ.get("ALLOW_REMOTE_MODEL_DOWNLOAD", "0") == "1"

PROMPT_MAX_LENGTH = 512
# Dev responses are about 18-29 words for the 10th-90th percentile, median about 23 words.
MIN_NEW_TOKENS = int(os.environ.get("MIN_NEW_TOKENS", "32"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "72"))
RESPONSE_MIN_WORDS = int(os.environ.get("RESPONSE_MIN_WORDS", "18"))
RESPONSE_TARGET_WORDS = int(os.environ.get("RESPONSE_TARGET_WORDS", "24"))
RESPONSE_MAX_WORDS = int(os.environ.get("RESPONSE_MAX_WORDS", "32"))
TRACK1_MAX_LENGTH = 512
LOAD_IN_4BIT = True
BF16 = True

# 32G 显存下提高默认 batch 来更充分利用显存；如果 OOM，把 PPO_BATCH_SIZE/GENERATION_BATCH_SIZE 降回 4/2。
PPO_BATCH_SIZE = int(os.environ.get("PPO_BATCH_SIZE", "8"))
PPO_MINI_BATCH_SIZE = int(os.environ.get("PPO_MINI_BATCH_SIZE", "2"))
PPO_GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("PPO_GRADIENT_ACCUMULATION_STEPS", "1"))
PPO_EPOCHS = int(os.environ.get("PPO_EPOCHS", "1"))
GENERATION_BATCH_SIZE = int(os.environ.get("GENERATION_BATCH_SIZE", "4"))
ATTN_IMPLEMENTATION = os.environ.get("ATTN_IMPLEMENTATION", "eager")
FORCE_POLICY_ON_SINGLE_GPU = os.environ.get("FORCE_POLICY_ON_SINGLE_GPU", "1") == "1"
TRACK1_DEVICE = os.environ.get("TRACK1_DEVICE", "cpu")
# 32G GPU 上单独 reference model 会再占一份 9B 显存，默认关闭。
USE_SEPARATE_REF_MODEL = os.environ.get("USE_SEPARATE_REF_MODEL", "0") == "1"

print("LOCAL_TRACK2_BASE_MODEL:", LOCAL_TRACK2_BASE_MODEL)
print("ALLOW_REMOTE_MODEL_DOWNLOAD:", ALLOW_REMOTE_MODEL_DOWNLOAD)
print("FORCE_POLICY_ON_SINGLE_GPU:", FORCE_POLICY_ON_SINGLE_GPU)
print("TRACK1_DEVICE:", TRACK1_DEVICE)
print("PPO_BATCH_SIZE:", PPO_BATCH_SIZE)
print("PPO_MINI_BATCH_SIZE:", PPO_MINI_BATCH_SIZE)
print("GENERATION_BATCH_SIZE:", GENERATION_BATCH_SIZE)
print("generation tokens:", MIN_NEW_TOKENS, "to", MAX_NEW_TOKENS)
print("target response words:", RESPONSE_MIN_WORDS, RESPONSE_TARGET_WORDS, RESPONSE_MAX_WORDS)


LOCAL_TRACK2_BASE_MODEL: /root/autodl-fs/nlpcc_models/Qwen3.5-9B
ALLOW_REMOTE_MODEL_DOWNLOAD: False
FORCE_POLICY_ON_SINGLE_GPU: True
TRACK1_DEVICE: cpu
PPO_BATCH_SIZE: 8
PPO_MINI_BATCH_SIZE: 2
GENERATION_BATCH_SIZE: 4
generation tokens: 32 to 72
target response words: 18 24 32


In [4]:
import json
import re
VALUE_LABELS = [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance",
]

label2id = {label : i for i, label in enumerate(VALUE_LABELS)}
id2label = {i : label for i, label in enumerate(VALUE_LABELS)}

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def get_target_value(prompt):
    value = prompt.split("\n\nTarget value:\n", 1)[1]
    value = value.split("\n\nTarget value definition:\n", 1)[0]
    return normalize_space(value)

def response_word_count(text):
    return len(re.findall(r"\b[\w'-]+\b", normalize_space(text)))

train_rows = read_jsonl(TRAIN_DATA_FILE)
dev_rows = read_jsonl(DEV_DATA_FILE)
reference_lengths = [response_word_count(row["response"]) for row in dev_rows if row.get("response")]
if reference_lengths:
    sorted_lengths = sorted(reference_lengths)
    p10 = sorted_lengths[int(0.10 * (len(sorted_lengths) - 1))]
    p50 = sorted_lengths[int(0.50 * (len(sorted_lengths) - 1))]
    p90 = sorted_lengths[int(0.90 * (len(sorted_lengths) - 1))]
    print("dev response word length p10/p50/p90:", p10, p50, p90)

ppo_rows = []
for i, row in enumerate(train_rows):
    prompt = row["prompt"]
    value = get_target_value(prompt)
    ppo_rows.append({
        "idx" : i,
        "prompt" : prompt,
        "target_value" : value,
    })

print("train_rows:", len(train_rows))
print("dev_rows:", len(dev_rows))
print("ppo_rows:", len(ppo_rows))
print("sample target:", ppo_rows[0]["target_value"])
print("sample prompt:", ppo_rows[0]["prompt"][:700])

dev response word length p10/p50/p90: 18 23 29
train_rows: 3520
dev_rows: 514
ppo_rows: 3520
sample target: Conformity–interpersonal
sample prompt: You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



#### load tokenizer and dataset

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer
# 加载tokenizer
tokenizer_scr = str(SFT_MODEL_DIR)
ppo_tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_scr, 
    trust_remote_code = True,
    )
if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
ppo_tokenizer.padding_side = "left"

# 数据类型转化
ppo_dataset = Dataset.from_list(ppo_rows)

# 数据编码与映射
def tokenizer_ppo_batch(batch):
    encoded = ppo_tokenizer(
        batch["prompt"],
        truncation = True,
        max_length = PROMPT_MAX_LENGTH,
        padding = False,
    )
    batch["input_ids"] = encoded["input_ids"]
    batch["attention_mask"] = encoded["attention_mask"]
    return batch

ppo_dataset = ppo_dataset.map(tokenizer_ppo_batch, batched=True)

print("ppo dataset:", len(ppo_dataset))
print("pad_token:", ppo_tokenizer.pad_token, ppo_tokenizer.pad_token_id)
print("sample input length:", len(ppo_dataset[0]["input_ids"]))
print("sample target:", ppo_dataset[0]["target_value"])
print("sample row prompt:", ppo_dataset[0]["prompt"])
print("sample prompt input_ids:", ppo_dataset[0]["input_ids"])

/root/miniconda3/envs/nlpcc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 3520/3520 [00:00<00:00, 11749.02 examples/s]

ppo dataset: 3520
pad_token: <|endoftext|> 248044
sample input length: 105
sample target: Conformity–interpersonal
sample row prompt: You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:

sample prompt input_ids: [2523, 513, 2574, 264, 14611, 11, 264, 3296, 11, 321, 264, 2100, 3611, 869, 13, 19208, 799, 61446, 11, 21716, 1965, 421, 10926, 279, 3296, 11, 17759, 279, 14611, 11, 321, 17185, 5117, 82, 440, 279, 2100, 869, 13, 271, 52217, 25, 198, 2523, 944, 303, 264, 12501, 6047, 1332, 678, 6436, 13418, 84270, 7077, 16981, 13, 271, 14162, 25, 1

#### load track1 model

In [6]:
# 加载track1的分类模型
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel
import torch

def load_track1_model(model_dir):
    model_dir = Path(model_dir)
    track1_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote=True)
    kwargs = dict(
        num_labels = len(VALUE_LABELS),
        id2label = id2label,
        label2id = label2id,
        trust_remote_code = True,
    )
    if(model_dir / "adapter_config.json").exists():
        base_model = AutoModelForSequenceClassification.from_pretrained(
            TRACK1_BASE_MODEL,
            **kwargs,
        )
        track1_model = PeftModel.from_pretrained(base_model, str(model_dir))
    else:
        track1_model = AutoModelForSequenceClassification.from_pretrained(
            str(model_dir),
            **kwargs,
        )
    for p in track1_model.parameters():
        p.requires_grad = False
    requested_device = TRACK1_DEVICE.lower()
    if requested_device == "cuda" and torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
    track1_model.to(device)
    track1_model.eval()
    return track1_model, track1_tokenizer, device

print("loading track1 model.....")
track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)
print("loaded on", track1_device)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


loading track1 model.....


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6254.91it/s]

loaded on cpu


#### design reward model(function)

In [7]:
import numpy as np
@torch.no_grad()
def compute_rewards(responses, target_values):
    texts = ["Response: " + normalize_space(r) for r in responses]
    inputs = track1_tokenizer(
        texts,
        max_length = TRACK1_MAX_LENGTH,
        truncation = True,
        padding = True,
        return_tensors = "pt",
    )    
    inputs = {k:v.to(track1_device) for k, v in inputs.items()}
    output = track1_model(**inputs)
    logits = output.logits.float()
    probs = torch.softmax(logits, dim = -1).cpu().numpy()

    rewards = []
    records = []
    
    for response, target_value, prob in zip(responses, target_values, probs):
        target_id = label2id[target_value]
        pred_id = int(prob.argmax())

        target_prob = float(prob[target_id])
        pred_prob = float(prob[pred_id])

        sorted_ids = np.argsort(prob)[: :-1]
        best_other_id = next(i for i in sorted_ids if i != target_id)

        margin = float(prob[target_id] - prob[best_other_id])

        word_count = response_word_count(response)
        if word_count < RESPONSE_MIN_WORDS:
            length_penalty = min(0.35, 0.035 * (RESPONSE_MIN_WORDS - word_count))
        elif word_count > RESPONSE_MAX_WORDS:
            length_penalty = min(0.35, 0.025 * (word_count - RESPONSE_MAX_WORDS))
        else:
            distance = abs(word_count - RESPONSE_TARGET_WORDS)
            length_penalty = 0.006 * distance

        length_bonus = 0.08 if RESPONSE_MIN_WORDS <= word_count <= RESPONSE_MAX_WORDS else 0.0
        raw_reward = target_prob + 0.25 * margin + length_bonus - length_penalty
        reward = 10 * raw_reward
        reward = float(np.clip(reward, -5.0, 5.0))
        rewards.append(torch.tensor(reward, dtype = torch.float32))
        records.append({
            "target_value": target_value,
            "pred_value":id2label[pred_id],
            "response":response,
            "target_prob": target_prob,
            "pred_prob": pred_prob,
            "word_count": word_count,
            "length_penalty": length_penalty,
            "reward":reward,
        })
    return rewards, records


#### load policy model

In [8]:
import sys
import torch
import transformers

def patch_transformers_for_legacy_trl():
    if hasattr(transformers, "top_k_top_p_filtering"):
        return
    try:
        from transformers.generation.utils import top_k_top_p_filtering
    except ImportError:
        def top_k_top_p_filtering(logits, top_k=0, top_p=1.0, filter_value=-float("Inf"), min_tokens_to_keep=1):
            if top_k > 0:
                top_k = min(max(top_k, min_tokens_to_keep), logits.size(-1))
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits = logits.masked_fill(indices_to_remove, filter_value)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                if min_tokens_to_keep > 1:
                    sorted_indices_to_remove[..., :min_tokens_to_keep] = 0
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, filter_value)
            return logits

    transformers.top_k_top_p_filtering = top_k_top_p_filtering

patch_transformers_for_legacy_trl()
for module_name in list(sys.modules):
    if module_name == "trl" or module_name.startswith("trl."):
        sys.modules.pop(module_name, None)

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import AutoModelForCausalLMWithValueHead
import gc

def resolve_track2_base_model():
    candidates = [
        LOCAL_TRACK2_BASE_MODEL,
        Path(TRACK2_BASE_MODEL),
        ROOT_DIR / "models" / TRACK2_BASE_MODEL.split("/")[-1],
        ROOT_DIR / "models" / TRACK2_BASE_MODEL.replace("/", "--"),
    ]
    for candidate in candidates:
        if candidate.exists() and (candidate / "config.json").exists():
            print("using local track2 base model:", candidate)
            return str(candidate)

    if ALLOW_REMOTE_MODEL_DOWNLOAD:
        print(f"local base model not found; downloading/using cache for {TRACK2_BASE_MODEL}")
        return TRACK2_BASE_MODEL

    raise FileNotFoundError(
        "Local Qwen3.5-9B base model was not found.\n"
        f"Expected local base model at: {LOCAL_TRACK2_BASE_MODEL}\n"
        "Please check that this directory contains config.json and model.safetensors shards. "
        "If you intentionally want remote download, set ALLOW_REMOTE_MODEL_DOWNLOAD=1."
    )

def build_ppo_model_kwargs():
    compute_dtype = torch.bfloat16 if BF16 else torch.float16
    model_kwargs = {
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
        "attn_implementation": ATTN_IMPLEMENTATION,
        "local_files_only": not ALLOW_REMOTE_MODEL_DOWNLOAD,
    }
    if torch.cuda.is_available() and FORCE_POLICY_ON_SINGLE_GPU:
        device_map = {"": torch.cuda.current_device()}
    else:
        device_map = "auto"

    if LOAD_IN_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = device_map
    else:
        model_kwargs["torch_dtype"] = compute_dtype
        model_kwargs["device_map"] = device_map
    return model_kwargs

def prepare_ppo_base_model():
    base_model = AutoModelForCausalLM.from_pretrained(
        resolve_track2_base_model(),
        **build_ppo_model_kwargs(),
    )
    if base_model.get_input_embeddings().num_embeddings != len(ppo_tokenizer):
        base_model.resize_token_embeddings(len(ppo_tokenizer))
    base_model.config.pad_token_id = ppo_tokenizer.pad_token_id
    base_model.config.use_cache = False

    if LOAD_IN_4BIT:
        base_model = prepare_model_for_kbit_training(base_model)
    return base_model

def attach_lora_to_value_head_model(value_head_model, model_dir, is_trainable, adapter_load_kwargs=None):
    adapter_load_kwargs = adapter_load_kwargs or {}
    value_head_model.pretrained_model = PeftModel.from_pretrained(
        value_head_model.pretrained_model,
        str(model_dir),
        is_trainable=is_trainable,
        **adapter_load_kwargs,
    )
    value_head_model.pretrained_model.config.use_cache = False
    value_head_model.is_peft_model = True
    return value_head_model

def infer_hidden_size_for_value_head(base_model):
    config = base_model.config
    for attr in ["hidden_size", "summary_hidden_size", "word_embed_proj_dim", "n_embd", "d_model"]:
        value = getattr(config, attr, None)
        if isinstance(value, int) and value > 0:
            return value

    for nested_attr in ["text_config", "decoder"]:
        nested_config = getattr(config, nested_attr, None)
        if nested_config is None:
            continue
        for attr in ["hidden_size", "summary_hidden_size", "word_embed_proj_dim", "n_embd", "d_model"]:
            value = getattr(nested_config, attr, None)
            if isinstance(value, int) and value > 0:
                return value

    input_embeddings = base_model.get_input_embeddings()
    if input_embeddings is not None and hasattr(input_embeddings, "weight"):
        return int(input_embeddings.weight.shape[1])

    output_embeddings = base_model.get_output_embeddings()
    if output_embeddings is not None:
        in_features = getattr(output_embeddings, "in_features", None)
        if isinstance(in_features, int) and in_features > 0:
            return in_features

    raise AttributeError(
        "Cannot infer hidden_size for TRL ValueHead from config or embeddings; "
        f"config type: {type(config).__name__}"
    )

def build_value_head_model_from_base(base_model):
    hidden_size = infer_hidden_size_for_value_head(base_model)
    base_model.config.hidden_size = hidden_size
    if not hasattr(base_model.config, "is_encoder_decoder"):
        base_model.config.is_encoder_decoder = False
    print(f"ValueHead hidden_size={hidden_size}")
    return AutoModelForCausalLMWithValueHead.from_pretrained(base_model)

def load_ppo_policy_model(model_dir):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    base_model = prepare_ppo_base_model()
    policy_model = build_value_head_model_from_base(base_model)
    policy_model = attach_lora_to_value_head_model(
        policy_model,
        model_dir,
        is_trainable=True,
    )
    return policy_model

policy_model = load_ppo_policy_model(SFT_MODEL_DIR)
print("policy model loaded")

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


using local track2 base model: /root/autodl-fs/nlpcc_models/Qwen3.5-9B


Loading weights: 100%|██████████| 427/427 [00:05<00:00, 81.10it/s] 


ValueHead hidden_size=4096
policy model loaded


#### load reference model

In [9]:
import gc

def load_ppo_ref_model(model_dir):
    if not USE_SEPARATE_REF_MODEL:
        print("USE_SEPARATE_REF_MODEL=0: skip loading a second 9B reference model to save VRAM.")
        return None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    base_model = prepare_ppo_base_model()
    ref_model = build_value_head_model_from_base(base_model)
    ref_model = attach_lora_to_value_head_model(
        ref_model,
        model_dir,
        is_trainable=False,
    )
    ref_model.eval()

    for p in ref_model.parameters():
        p.requires_grad = False
    return ref_model

ref_model = load_ppo_ref_model(SFT_MODEL_DIR)
print("reference model:", "disabled" if ref_model is None else "loaded")

USE_SEPARATE_REF_MODEL=0: skip loading a second 9B reference model to save VRAM.
reference model: disabled


#### PPO config and Trainer

In [10]:
import inspect
import torch
import transformers

def patch_transformers_for_legacy_trl():
    if hasattr(transformers, "top_k_top_p_filtering"):
        return
    try:
        from transformers.generation.utils import top_k_top_p_filtering
    except ImportError:
        def top_k_top_p_filtering(logits, top_k=0, top_p=1.0, filter_value=-float("Inf"), min_tokens_to_keep=1):
            if top_k > 0:
                top_k = min(max(top_k, min_tokens_to_keep), logits.size(-1))
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits = logits.masked_fill(indices_to_remove, filter_value)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                if min_tokens_to_keep > 1:
                    sorted_indices_to_remove[..., :min_tokens_to_keep] = 0
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, filter_value)
            return logits

    transformers.top_k_top_p_filtering = top_k_top_p_filtering

patch_transformers_for_legacy_trl()

from trl import PPOConfig, PPOTrainer

def assert_legacy_ppo_trainer():
    params = inspect.signature(PPOTrainer).parameters
    required = {
        name for name, param in params.items()
        if param.default is inspect._empty
        and param.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)
        and name != "self"
    }
    if {"reward_model", "value_model"} & required:
        raise RuntimeError(
            "当前环境安装的是新版 TRL PPOTrainer，需要 reward_model/value_model；"
            "本 notebook 使用的是旧版 TRL 的 AutoModelForCausalLMWithValueHead + ppo_trainer.step(...) 写法。\n"
            "请在服务器执行：pip install 'trl==0.7.11'，然后重启 kernel 从头运行。"
        )

assert_legacy_ppo_trainer()

def ppo_collator(features):
    batch = {}
    for key in features[0].keys():
        values = [feature[key] for feature in features]
        if key in ["input_ids", "attention_mask"]:
            values = [torch.tensor(v, dtype=torch.long) for v in values]
        
        batch[key] = values
    return batch

def build_ppo_config():
    backward_batch_size = PPO_MINI_BATCH_SIZE * PPO_GRADIENT_ACCUMULATION_STEPS
    if PPO_BATCH_SIZE % backward_batch_size != 0:
        raise ValueError(
            f"PPO_BATCH_SIZE={PPO_BATCH_SIZE} must be a multiple of "
            f"PPO_MINI_BATCH_SIZE * PPO_GRADIENT_ACCUMULATION_STEPS={backward_batch_size}"
        )

    params = inspect.signature(PPOConfig).parameters
    kwargs = {
        "model_name": str(SFT_MODEL_DIR),
        "learning_rate": 1e-6,
        "batch_size": PPO_BATCH_SIZE,
        "mini_batch_size": PPO_MINI_BATCH_SIZE,
        "gradient_accumulation_steps": PPO_GRADIENT_ACCUMULATION_STEPS,
        "remove_unused_columns": False,
        "target_kl": 0.1,
        "log_with": None,
    }
    if "num_ppo_epochs" in params:
        kwargs["num_ppo_epochs"] = PPO_EPOCHS
    elif "ppo_epochs" in params:
        kwargs["ppo_epochs"] = PPO_EPOCHS

    supported_kwargs = {key: value for key, value in kwargs.items() if key in params}
    skipped_kwargs = sorted(set(kwargs) - set(supported_kwargs))
    if skipped_kwargs:
        print("PPOConfig skipped unsupported args:", skipped_kwargs)
    print(
        "PPO speed config:",
        f"batch_size={PPO_BATCH_SIZE},",
        f"mini_batch_size={PPO_MINI_BATCH_SIZE},",
        f"gradient_accumulation_steps={PPO_GRADIENT_ACCUMULATION_STEPS},",
        f"ppo_epochs={PPO_EPOCHS}",
    )
    return PPOConfig(**supported_kwargs)

ppo_config = build_ppo_config()
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=ppo_tokenizer,
    dataset=ppo_dataset,
    data_collator=ppo_collator,
)
print("num batches:", len(ppo_trainer.dataloader))

PPO speed config: batch_size=8, mini_batch_size=2, gradient_accumulation_steps=1, ppo_epochs=1
num batches: 440


#### begin training

In [11]:
from tqdm.auto import tqdm
generation_kwargs = {
    "min_new_tokens": MIN_NEW_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample" : True,
    "temperature" : 0.8,
    "top_p" : 0.92,
    "repetition_penalty": 1.05,
    "no_repeat_ngram_size": 4,
    "use_cache" : True,
    "pad_token_id" : ppo_tokenizer.pad_token_id,
    "eos_token_id" : ppo_tokenizer.eos_token_id,
}

def iter_wrapped_models(model):
    seen = set()
    stack = [model]
    while stack:
        obj = stack.pop()
        if obj is None or id(obj) in seen:
            continue
        seen.add(id(obj))
        yield obj
        for attr in ["module", "pretrained_model", "base_model", "model"]:
            stack.append(getattr(obj, attr, None))

def set_gradient_checkpointing(model, enabled):
    method_name = "gradient_checkpointing_enable" if enabled else "gradient_checkpointing_disable"
    for obj in iter_wrapped_models(model):
        method = getattr(obj, method_name, None)
        if callable(method):
            try:
                method()
            except TypeError:
                pass

def set_use_cache(model, enabled):
    for obj in iter_wrapped_models(model):
        config = getattr(obj, "config", None)
        if config is not None and hasattr(config, "use_cache"):
            config.use_cache = enabled

print("generation batch_size:", GENERATION_BATCH_SIZE)
print("generation token range:", MIN_NEW_TOKENS, "to", MAX_NEW_TOKENS)
print("target response word range:", RESPONSE_MIN_WORDS, "to", RESPONSE_MAX_WORDS, "target", RESPONSE_TARGET_WORDS)
for step, batch in enumerate(tqdm(ppo_trainer.dataloader, desc="ppo training")):
    query_tensors = batch["input_ids"]
    if "target_value" not in batch:
        raise KeyError(f"target_value missing from PPO batch. Available keys: {list(batch.keys())}")
    target_values = batch["target_value"]

    policy_model.eval()
    set_gradient_checkpointing(policy_model, False)
    set_use_cache(policy_model, True)
    try:
        response_tensors = ppo_trainer.generate(
            query_tensors, 
            batch_size = GENERATION_BATCH_SIZE,
            return_prompt = False,
            **generation_kwargs,
        )
    finally:
        set_use_cache(policy_model, False)
        set_gradient_checkpointing(policy_model, True)
        policy_model.train()
    responses = ppo_tokenizer.batch_decode(
        response_tensors,
        skip_special_tokens = True,
    )
    responses = [normalize_space(r) for r in responses]
    rewards, reward_records = compute_rewards(
        responses = responses,
        target_values = target_values,
    )
    stats = ppo_trainer.step(
        query_tensors,
        response_tensors,
        rewards,
    )
    reward_num = [value.item() for value in rewards]
    mean_reward = float(np.mean(reward_num))
    mean_words = float(np.mean([record["word_count"] for record in reward_records]))
    if step % 10 == 0:
        print("=" * 80)
        print("step:", step)
        print("mean reward:", round(mean_reward, 4))
        print("mean words:", round(mean_words, 2))
        record = reward_records[0]
        print("target:", record["target_value"])
        print("pred:", record["pred_value"])
        print("target_prob:", round(record["target_prob"], 4))
        print("words:", record["word_count"])
        print("length_penalty:", round(record["length_penalty"], 4))
        print("reward:", round(record["reward"], 4))
        print("response:", record["response"])

policy_model.save_pretrained(str(OUTPUTS_DIR))
ppo_tokenizer.save_pretrained(str(OUTPUTS_DIR))

print("saved PPO model to:", OUTPUTS_DIR)


generation batch_size: 4
generation token range: 32 to 72
target response word range: 18 to 32 target 24


ppo training:   0%|          | 1/440 [00:16<2:00:50, 16.52s/it]

step: 0
mean reward: 2.9384
mean words: 37.25
target: Conformity–interpersonal
pred: Conformity–interpersonal
target_prob: 0.8365
words: 28
length_penalty: 0.024
reward: 5.0
response: I would give them my jacket without hesitation, reassuring them that I trust them to take care of it, prioritizing our friendship over the risk of potential conflict.


ppo training:   2%|▎         | 11/440 [02:55<1:54:14, 15.98s/it]

step: 10
mean reward: 5.0
mean words: 33.5
target: Conformity–interpersonal
pred: Conformity–interpersonal
target_prob: 0.8069
words: 23
length_penalty: 0.006
reward: 5.0
response: Apologize sincerely, acknowledge their contribution, and adjust my behavior to ensure we stay united and avoid future conflicts.user user You are given


...

step: 420
mean reward: 4.3542
mean words: 30.5
target: Universalism–concern
pred: Universalism–concern
target_prob: 0.88
words: 29
length_penalty: 0.03
reward: 5.0
response: I would prioritize the student's dignity by addressing the bullying privately, emphasize respect for individuality, and actively foster an inclusive classroom environment where every student feels safe and valued.


ppo training:  98%|█████████▊| 431/440 [1:34:18<01:54, 12.76s/it]

step: 430
mean reward: 4.1471
mean words: 27.62
target: Stimulation
pred: Stimulation
target_prob: 0.9841
words: 24
length_penalty: 0.0
reward: 5.0
response: I would seek out innovative projects or propose new initiatives to inject novelty into my routine, ensuring I stay engaged and challenged.user assistantuser


ppo training: 100%|██████████| 440/440 [1:36:09<00:00, 13.11s/it]
/root/miniconda3/envs/nlpcc/lib/python3.11/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


saved PPO model to: /root/autodl-tmp/nlpcc_task2/outputs/track2/qwen35_9b/ppo
